# ETL_padroniza_bairros

This notebook standardizes neighborhood names before geocoding.

It preserves the original `dim_bairro` values, creates normalized comparison keys, applies alias rules, performs exact matching against a canonical reference table, and only then applies fuzzy matching for unresolved records.


## 1. Setup

Define catalog/schema names, input CSV paths, output table names, and matching thresholds.

In [0]:
# =========================
# 1. Setup
# =========================

from pyspark.sql.functions import (
    col, lower, trim, regexp_replace, translate, length,
    levenshtein, greatest, row_number, when, lit, coalesce, count
)
from pyspark.sql.window import Window

CATALOG = "workspace"
SCHEMA = "default"
TABLE_PREFIX = f"{CATALOG}.{SCHEMA}"

SOURCE_DIM_BAIRRO_TABLE = f"{TABLE_PREFIX}.dim_bairro"

REF_CANONICAL_TABLE = f"{TABLE_PREFIX}.ref_bairros_canonicos"
REF_ALIAS_TABLE = f"{TABLE_PREFIX}.ref_bairros_alias"
REF_STANDARDIZED_TABLE = f"{TABLE_PREFIX}.ref_bairro_padronizado"
REF_GEOCODING_INPUT_TABLE = f"{TABLE_PREFIX}.ref_bairros_para_geocoding"

# Thresholds used only after alias and exact matching fail.
SIMILARITY_THRESHOLD = 0.85
SIMILARITY_GAP_THRESHOLD = 0.05

## 2. Text normalization helper

This function creates a comparison key. It is not intended to replace the final neighborhood name shown to the user.

Examples:

- `Água Branca I`, `agua branca i`, `Água Branca 1` -> `agua branca 1`
- `jd brasil` -> `jardim brasil`
- `n.s.apda` -> `nossa senhora aparecida`

In [0]:
# =========================
# 2. Text normalization helper
# =========================

def normalize_neighborhood_key(c):
    """Return a normalized Spark column for neighborhood matching."""

    x = lower(trim(c))

    # Fix common encoding/mojibake issues found in the source data.
    x = regexp_replace(x, "ã§", "c")
    x = regexp_replace(x, "ã£", "a")
    x = regexp_replace(x, "ã¡", "a")
    x = regexp_replace(x, "ã¢", "a")
    x = regexp_replace(x, "ã©", "e")
    x = regexp_replace(x, "ãª", "e")
    x = regexp_replace(x, "ã­", "i")
    x = regexp_replace(x, "ã³", "o")
    x = regexp_replace(x, "ã´", "o")
    x = regexp_replace(x, "ãµ", "o")
    x = regexp_replace(x, "ãº", "u")
    x = regexp_replace(x, "ã¼", "u")
    x = regexp_replace(x, "妹", "ca")

    # Remove accents for matching only.
    x = translate(
        x,
        "áàãâäéèêëíìîïóòõôöúùûüç",
        "aaaaaeeeeiiiiooooouuuuc"
    )

    # Remove punctuation and normalize spaces.
    x = regexp_replace(x, r"[^a-z0-9\s]", " ")
    x = regexp_replace(x, r"\s+", " ")
    x = trim(x)

    # Expand common abbreviations.
    x = regexp_replace(x, r"\bjd\b", "jardim")
    x = regexp_replace(x, r"\bjdm\b", "jardim")
    x = regexp_replace(x, r"\bvl\b", "vila")
    x = regexp_replace(x, r"\bres\b", "residencial")
    x = regexp_replace(x, r"\bcond\b", "condominio")
    x = regexp_replace(x, r"\bch\b", "chacara")
    x = regexp_replace(x, r"\bchac\b", "chacara")
    x = regexp_replace(x, r"\bconj\b", "conjunto")
    x = regexp_replace(x, r"\bhab\b", "habitacional")

    # Expand common religious/name abbreviations.
    x = regexp_replace(x, r"\bn s apda\b", "nossa senhora aparecida")
    x = regexp_replace(x, r"\bns apda\b", "nossa senhora aparecida")
    x = regexp_replace(x, r"\bnossa senhora apda\b", "nossa senhora aparecida")

    # Normalize roman numerals to Arabic numbers for comparison only.
    # Important: replace iii before ii before i.
    x = regexp_replace(x, r"\biii\b", "3")
    x = regexp_replace(x, r"\bii\b", "2")
    x = regexp_replace(x, r"\bi\b", "1")

    # Normalize spaces again after replacements.
    x = regexp_replace(x, r"\s+", " ")
    x = trim(x)

    return x

## 3. Load source neighborhoods from `dim_bairro`

The original neighborhood name is preserved as `bairro_original`.

In [0]:
# =========================
# 3. Load source neighborhoods from dim_bairro
# =========================

df_bairros = (
    spark.table(SOURCE_DIM_BAIRRO_TABLE)
    .select(
        col("id_bairro"),
        col("bairro").alias("bairro_original")
    )
    .dropDuplicates(["id_bairro"])
)

print(f"Source neighborhoods loaded: {df_bairros.count()}")
display(df_bairros.orderBy("bairro_original"))

## 4. Load canonical and alias reference files

The canonical file contains the standardized neighborhood/locality names. The alias file contains known variations that should be solved before fuzzy matching.

In [0]:
# =========================
# 4. Load canonical and alias reference tables
# =========================

df_canonical_raw = spark.table("workspace.default.canonical_neighborhoods")

df_alias_raw = spark.table("workspace.default.alias_seed_rules")

required_canonical_cols = {
    "id_bairro_canonico",
    "bairro_canonico",
    "bairro_canonico_norm",
    "query_geocoding"
}

required_alias_cols = {
    "alias_norm_observado",
    "bairro_canonico_sugerido"
}

missing_canonical = required_canonical_cols - set(df_canonical_raw.columns)
missing_alias = required_alias_cols - set(df_alias_raw.columns)

if missing_canonical:
    raise ValueError(f"Missing columns in canonical table: {missing_canonical}")

if missing_alias:
    raise ValueError(f"Missing columns in alias table: {missing_alias}")

df_canonical_raw = df_canonical_raw.select(
    "id_bairro_canonico",
    "bairro_canonico",
    "bairro_canonico_norm",
    "municipio",
    "uf",
    "pais",
    "tipo_localidade",
    "use_for_geocoding",
    "query_geocoding",
    "source_primary_url",
    "source_secondary_url",
    "observacao"
)

df_alias_raw = df_alias_raw.select(
    "alias_norm_observado",
    "bairro_canonico_sugerido",
    "observacao"
)

print(f"Canonical records loaded: {df_canonical_raw.count()}")
print(f"Alias records loaded: {df_alias_raw.count()}")

display(df_canonical_raw.orderBy("bairro_canonico"))
display(df_alias_raw.orderBy("alias_norm_observado"))

## 5. Create normalized keys and save reference tables

This step prepares the canonical and alias tables for exact joins.

In [0]:
# =========================
# 5. Create normalized keys and save reference tables
# =========================

df_bairros_norm = (
    df_bairros
    .withColumn("bairro_original_key", normalize_neighborhood_key(col("bairro_original")))
)

df_canonical = (
    df_canonical_raw
    .withColumn("bairro_canonico_key", normalize_neighborhood_key(col("bairro_canonico")))
    .dropDuplicates(["bairro_canonico_key"])
)

df_alias = (
    df_alias_raw
    .withColumn("alias_key", normalize_neighborhood_key(col("alias_norm_observado")))
    .withColumn("bairro_canonico_sugerido_key", normalize_neighborhood_key(col("bairro_canonico_sugerido")))
    .dropDuplicates(["alias_key"])
)

# Save reference tables in Delta format for reproducibility and easier reuse by other notebooks.
df_canonical.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_CANONICAL_TABLE)
df_alias.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_ALIAS_TABLE)

print(f"Saved canonical reference table: {REF_CANONICAL_TABLE}")
print(f"Saved alias reference table: {REF_ALIAS_TABLE}")

display(df_bairros_norm.orderBy("bairro_original_key"))

## 6. Apply alias and exact matches

Alias rules are applied first because they solve known variants and abbreviations. Exact matching against the canonical reference is applied next.

In [0]:
# =========================
# 6. Apply alias and exact matches
# =========================

# 6.1 Alias match
alias_joined = (
    df_bairros_norm
    .join(
        df_alias,
        df_bairros_norm["bairro_original_key"] == df_alias["alias_key"],
        "left"
    )
)

alias_resolved = (
    alias_joined
    .join(
        df_canonical,
        alias_joined["bairro_canonico_sugerido_key"] == df_canonical["bairro_canonico_key"],
        "left"
    )
    .select(
        alias_joined["id_bairro"],
        alias_joined["bairro_original"],
        alias_joined["bairro_original_key"],
        col("bairro_canonico").alias("bairro_by_alias"),
        col("id_bairro_canonico").alias("id_bairro_canonico_by_alias"),
        col("query_geocoding").alias("query_geocoding_by_alias")
    )
)

# 6.2 Exact match against canonical reference
exact_matched = (
    alias_resolved
    .join(
        df_canonical,
        alias_resolved["bairro_original_key"] == df_canonical["bairro_canonico_key"],
        "left"
    )
    .select(
        alias_resolved["id_bairro"],
        alias_resolved["bairro_original"],
        alias_resolved["bairro_original_key"],
        alias_resolved["bairro_by_alias"],
        alias_resolved["id_bairro_canonico_by_alias"],
        alias_resolved["query_geocoding_by_alias"],
        col("bairro_canonico").alias("bairro_by_exact"),
        col("id_bairro_canonico").alias("id_bairro_canonico_by_exact"),
        col("query_geocoding").alias("query_geocoding_by_exact")
    )
)

print("Alias and exact matching completed.")
display(exact_matched.orderBy("bairro_original"))

## 7. Fuzzy matching for unresolved records

Fuzzy matching is applied only to records that were not solved by alias or exact matching. A fuzzy match is accepted only when:

- the best similarity is greater than or equal to `SIMILARITY_THRESHOLD`; and
- the gap between the best and the second-best candidate is greater than or equal to `SIMILARITY_GAP_THRESHOLD`.

This reduces the risk of accepting ambiguous matches.

In [0]:
# =========================
# 7. Fuzzy matching for unresolved records
# =========================

unresolved = (
    exact_matched
    .filter(
        col("bairro_by_alias").isNull() &
        col("bairro_by_exact").isNull()
    )
    .select("id_bairro", "bairro_original", "bairro_original_key")
)

print(f"Records sent to fuzzy matching: {unresolved.count()}")

fuzzy_candidates = (
    unresolved
    .crossJoin(df_canonical)
    .withColumn(
        "distance",
        levenshtein(col("bairro_original_key"), col("bairro_canonico_key"))
    )
    .withColumn(
        "max_len",
        greatest(length(col("bairro_original_key")), length(col("bairro_canonico_key")))
    )
    .withColumn(
        "similarity",
        when(col("max_len") > 0, 1 - (col("distance") / col("max_len"))).otherwise(lit(0.0))
    )
)

rank_window = Window.partitionBy("id_bairro").orderBy(col("similarity").desc(), col("distance").asc())

fuzzy_ranked = (
    fuzzy_candidates
    .withColumn("match_rank", row_number().over(rank_window))
    .filter(col("match_rank").isin(1, 2))
)

fuzzy_first = (
    fuzzy_ranked
    .filter(col("match_rank") == 1)
    .select(
        "id_bairro",
        col("bairro_canonico").alias("bairro_by_fuzzy"),
        col("id_bairro_canonico").alias("id_bairro_canonico_by_fuzzy"),
        col("query_geocoding").alias("query_geocoding_by_fuzzy"),
        col("similarity").alias("best_similarity"),
        col("distance").alias("best_distance")
    )
)

fuzzy_second = (
    fuzzy_ranked
    .filter(col("match_rank") == 2)
    .select(
        "id_bairro",
        col("bairro_canonico").alias("second_best_match"),
        col("similarity").alias("second_similarity")
    )
)

fuzzy_scored = (
    fuzzy_first
    .join(fuzzy_second, on="id_bairro", how="left")
    .withColumn(
        "similarity_gap",
        col("best_similarity") - coalesce(col("second_similarity"), lit(0.0))
    )
    .withColumn(
        "fuzzy_accepted",
        (col("best_similarity") >= lit(SIMILARITY_THRESHOLD)) &
        (col("similarity_gap") >= lit(SIMILARITY_GAP_THRESHOLD))
    )
)

display(fuzzy_scored.orderBy(col("best_similarity").asc()))

## 8. Consolidate standardized neighborhood names

The final standardized name is selected in this priority order:

1. alias match;
2. exact canonical match;
3. accepted fuzzy match;
4. unmatched.

In [0]:
# =========================
# 8. Consolidate standardized neighborhood names
# =========================

bairro_padronizado = (
    exact_matched
    .join(fuzzy_scored, on="id_bairro", how="left")
    .withColumn(
        "bairro_padronizado",
        coalesce(
            col("bairro_by_alias"),
            col("bairro_by_exact"),
            when(col("fuzzy_accepted") == True, col("bairro_by_fuzzy"))
        )
    )
    .withColumn(
        "id_bairro_canonico",
        coalesce(
            col("id_bairro_canonico_by_alias"),
            col("id_bairro_canonico_by_exact"),
            when(col("fuzzy_accepted") == True, col("id_bairro_canonico_by_fuzzy"))
        )
    )
    .withColumn(
        "query_geocoding",
        coalesce(
            col("query_geocoding_by_alias"),
            col("query_geocoding_by_exact"),
            when(col("fuzzy_accepted") == True, col("query_geocoding_by_fuzzy"))
        )
    )
    .withColumn(
        "match_method",
        when(col("bairro_by_alias").isNotNull(), lit("alias"))
        .when(col("bairro_by_exact").isNotNull(), lit("exact"))
        .when(col("fuzzy_accepted") == True, lit("fuzzy"))
        .otherwise(lit("unmatched"))
    )
    .withColumn(
        "needs_review",
        when(col("match_method") == "unmatched", lit(True)).otherwise(lit(False))
    )
    .select(
        "id_bairro",
        "bairro_original",
        "bairro_original_key",
        "id_bairro_canonico",
        "bairro_padronizado",
        "query_geocoding",
        "match_method",
        col("best_similarity").alias("similarity"),
        "second_best_match",
        "second_similarity",
        "similarity_gap",
        "needs_review"
    )
)

display(bairro_padronizado.orderBy("match_method", "bairro_original"))

## 9. Save standardized mapping and geocoding input

`ref_bairro_padronizado` maps each original `id_bairro` to a standardized canonical neighborhood.

`ref_bairros_para_geocoding` contains only unique standardized neighborhoods and should be used as the input for geocoding.

In [0]:
# =========================
# 9. Save standardized mapping and geocoding input
# =========================

bairro_padronizado.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    REF_STANDARDIZED_TABLE
)

bairros_para_geocoding = (
    bairro_padronizado
    .filter(col("id_bairro_canonico").isNotNull())
    .select(
        "id_bairro_canonico",
        col("bairro_padronizado").alias("bairro_canonico"),
        "query_geocoding"
    )
    .dropDuplicates(["id_bairro_canonico"])
    .orderBy("bairro_canonico")
)

bairros_para_geocoding.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    REF_GEOCODING_INPUT_TABLE
)

print(f"Saved standardized mapping table: {REF_STANDARDIZED_TABLE}")
print(f"Saved geocoding input table: {REF_GEOCODING_INPUT_TABLE}")

display(bairros_para_geocoding)

## 10. Validation summary

These checks help verify whether the standardization step produced usable results before geocoding.

In [0]:
# =========================
# 10. Validation summary
# =========================

print("Match summary:")
display(
    bairro_padronizado
    .groupBy("match_method", "needs_review")
    .agg(count("*").alias("records"))
    .orderBy("match_method", "needs_review")
)

print("Unmatched records:")
display(
    bairro_padronizado
    .filter(col("needs_review") == True)
    .orderBy("bairro_original")
)

print("Accepted fuzzy matches:")
display(
    bairro_padronizado
    .filter(col("match_method") == "fuzzy")
    .orderBy(col("similarity").asc())
)

print("Original names grouped under the same canonical neighborhood:")
display(
    bairro_padronizado
    .filter(col("id_bairro_canonico").isNotNull())
    .groupBy("id_bairro_canonico", "bairro_padronizado")
    .agg(count("*").alias("source_names"))
    .orderBy(col("source_names").desc(), "bairro_padronizado")
)

## 11. SQL smoke tests

Use these queries to inspect the generated tables after the notebook finishes.

In [0]:
# =========================
# 11. SQL smoke tests
# =========================

spark.sql(f"SELECT match_method, needs_review, COUNT(*) AS records FROM {REF_STANDARDIZED_TABLE} GROUP BY match_method, needs_review ORDER BY match_method, needs_review").show()
spark.sql(f"SELECT COUNT(*) AS neighborhoods_for_geocoding FROM {REF_GEOCODING_INPUT_TABLE}").show()